# Реализация PPO, PPO + World Model в дискретном пространстве

In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import random
import matplotlib.pyplot as plt
import imageio
from torch.distributions import Categorical
from collections import deque
from IPython.display import Image as IPyImage
from IPython.display import display, Image

## Гиперпараметры

In [ ]:
HP = {
    # Архитектура сети
    'hidden_size': 256, # было 128
    'activation': nn.Tanh,
    
    # Обучение агента
    'seeds': [42, 69],
    'max_episodes': 2500,
    'lr_policy': 1e-4,            # было 3e-4,
    'lr_wm': 5e-5,                # было 1e-4,
    'batch_size': 512,            # было 2048,
    'gamma': 0.999,               # было 0.99,
    'gae_lambda': 0.98,           # было 0.95,
    'clip_eps': 0.25,             # было 0.2,
    'ent_coef': 0.1,              # было 0.01,
    'n_epochs': 10,               # было 4

    # Модель мира
    'wm_train_steps': 3,          # Было 5
    'wm_batch_size': 256,         # Было 512
    'imagination_start': 100,     # Было 50
    'real_imag_ratio': 0.5,       # Соотношение реальных/синтетических данных
    'wm_horizon': 32,             # было 16, # Макс длина синтетических траекторий
    'done_loss_coef': 0.8,        #было 0.5, # Коэффициент для потери done
    
    # Нормализация
    'norm_eps': 1e-8,
    'grad_clip': 0.5,
    
    # Визуализация
    'plot_interval': 50,
    'gif_fps': 15,
    'render_frames': 300,
    'smooth_window': 50
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Вспомогательные классы и функции

In [ ]:
class RunningMeanStd:
    def __init__(self, shape):
        self.n = 0
        self.mean = np.zeros(shape)
        self.var = np.ones(shape)
    
    def update(self, x):
        x = np.asarray(x)
        m = np.nanmean(x, axis=0)
        v = np.nanvar(x, axis=0)
        count = x.shape[0]
        self.mean = (self.n * self.mean + count * m) / (self.n + count)
        self.var = (self.n * self.var + count * v) / (self.n + count)
        self.n += count

    def normalize(self, x):
        return (x - self.mean) / np.sqrt(self.var + HP['norm_eps'])

class DiscretePolicy(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, HP['hidden_size']),
            HP['activation'](),
            nn.Linear(HP['hidden_size'], HP['hidden_size']),
            HP['activation'](),
        )
        self.actor = nn.Linear(HP['hidden_size'], act_dim)
        self.critic = nn.Linear(HP['hidden_size'], 1)
        
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                nn.init.orthogonal_(layer.weight, gain=np.sqrt(2))
                nn.init.constant_(layer.bias, 0)
        nn.init.orthogonal_(self.actor.weight, gain=0.01)
        nn.init.orthogonal_(self.critic.weight, gain=1.0)

    def forward(self, x):
        x = self.net(torch.clamp(x, -10, 10))
        logits = self.actor(x)
        value = self.critic(x).squeeze(-1)
        return logits, value

class WorldModel(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.act_dim = act_dim
        self.net = nn.Sequential(
            nn.Linear(obs_dim + act_dim, HP['hidden_size']),
            HP['activation'](),
            nn.Linear(HP['hidden_size'], HP['hidden_size']),
            HP['activation'](),
        )
        self.next_state = nn.Linear(HP['hidden_size'], obs_dim)
        self.reward = nn.Linear(HP['hidden_size'], 1)
        self.done_head = nn.Sequential(
            nn.Linear(HP['hidden_size'], 1),
            nn.Sigmoid()
        )
        
        # Инициализация весов
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                nn.init.orthogonal_(layer.weight, gain=np.sqrt(2))
        nn.init.orthogonal_(self.next_state.weight, gain=1.0)
        nn.init.orthogonal_(self.reward.weight, gain=1.0)
        nn.init.orthogonal_(self.done_head[0].weight, gain=1.0)

    def forward(self, s, a):
        a_onehot = torch.nn.functional.one_hot(a, self.act_dim).float()
        h = self.net(torch.cat([s, a_onehot], dim=-1))
        ns = self.next_state(h)
        r = self.reward(h).squeeze(-1)
        d = self.done_head(h).squeeze(-1)
        return ns, r, d

class EpisodeBuffer:
    def __init__(self, max_episodes=100):
        self.buffer = deque(maxlen=max_episodes)
    
    def add_episode(self, states, actions, rewards, dones):
        self.buffer.append({
            'states': np.array(states),
            'actions': np.array(actions),
            'rewards': np.array(rewards),
            'dones': np.array(dones)
        })
    
    def sample_sequences(self, batch_size, seq_length):
        states, actions, rewards, dones, next_states = [], [], [], [], []
        
        for _ in range(batch_size):
            # Выбор случайного эпизода достаточной длины
            while True:
                ep = random.choice(self.buffer)
                ep_len = len(ep['states'])
                if ep_len >= seq_length + 1:
                    break
            
            # Выбор случайного стартового индекса
            start = np.random.randint(0, ep_len - seq_length)
            
            # Извлечение последовательности
            end = start + seq_length
            states.append(ep['states'][start:end])
            actions.append(ep['actions'][start:end])
            rewards.append(ep['rewards'][start:end])
            dones.append(ep['dones'][start:end])
            next_states.append(ep['states'][start+1:end+1])  # Исправленный срез
        
        return (
            torch.FloatTensor(np.array(states)),
            torch.LongTensor(np.array(actions)),
            torch.FloatTensor(np.array(rewards)),
            torch.FloatTensor(np.array(dones)),
            torch.FloatTensor(np.array(next_states)),
        )

## PPO (GAE + Update)

In [ ]:
def compute_gae(values, rewards, dones):
    adv = []
    last_adv = 0
    next_v = 0
    for t in reversed(range(len(rewards))):
        delta = rewards[t] + HP['gamma'] * next_v * (1 - dones[t]) - values[t]
        last_adv = delta + HP['gamma'] * HP['gae_lambda'] * (1 - dones[t]) * last_adv
        adv.insert(0, last_adv)
        next_v = values[t]
    return torch.tensor(adv, device=device), torch.tensor(values, device=device) + torch.tensor(adv, device=device)

def ppo_update(policy, optimizer, states, actions, old_log_probs, advantages, returns):
    for _ in range(HP['n_epochs']):
        logits, values = policy(states)
        dist = Categorical(logits=logits)
        new_log_probs = dist.log_prob(actions)
        
        ratio = (new_log_probs - old_log_probs).exp()
        clipped_ratio = torch.clamp(ratio, 1-HP['clip_eps'], 1+HP['clip_eps'])
        
        policy_loss = -torch.min(ratio * advantages, clipped_ratio * advantages).mean()
        value_loss = 0.5 * (values - returns).pow(2).mean()
        entropy_loss = -HP['ent_coef'] * dist.entropy().mean()
        
        loss = policy_loss + value_loss + entropy_loss
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), HP['grad_clip'])
        optimizer.step()

## Тренировка агента

In [ ]:
def train(env_name, seed=42, use_wm=False, save_gif=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    env = gym.make(env_name)
    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.n

    # Инициализация нормализации и политики
    obs_norm = RunningMeanStd(obs_dim)
    policy = DiscretePolicy(obs_dim, act_dim).to(device)
    optimizer = optim.Adam(policy.parameters(), lr=HP['lr_policy'])
    
    # Инициализация World Model и связанных компонентов
    wm, wm_opt, wm_obs_norm, episode_buffer = None, None, None, None
    if use_wm:
        wm = WorldModel(obs_dim, act_dim).to(device)
        wm_opt = optim.Adam(wm.parameters(), lr=HP['lr_wm'])
        wm_obs_norm = RunningMeanStd(obs_dim)
        episode_buffer = EpisodeBuffer(max_episodes=100)
    
    rewards = []
    steps = 0
    total_samples = 0

    for ep in range(HP['max_episodes']):
        # Сбор реальных данных
        real_buf = {'s': [], 'a': [], 'r': [], 'd': [], 'lp': []}
        ep_states, ep_actions, ep_rewards, ep_dones = [], [], [], []
        state, _ = env.reset(seed=seed+ep)
        ep_reward = 0
        
        while len(real_buf['r']) < HP['batch_size']:
            # Нормализация состояния
            s_n = obs_norm.normalize(state)
            st = torch.FloatTensor(s_n).to(device).unsqueeze(0)
            
            # Выбор действия
            with torch.no_grad():
                logits, value = policy(st)
                dist = Categorical(logits=logits)
                action = dist.sample()
                lp = dist.log_prob(action)
            
            # Шаг в среде
            next_state, reward, done, trunc, _ = env.step(action.item())
            done = done or trunc
            
            # Сохранение перехода
            real_buf['s'].append(s_n)
            real_buf['a'].append(action.item())
            real_buf['r'].append(reward)
            real_buf['d'].append(float(done))
            real_buf['lp'].append(lp.item())
            
            # Для WM: сохранение сырых состояний
            ep_states.append(state)
            ep_actions.append(action.item())
            ep_rewards.append(reward)
            ep_dones.append(done)
            
            # Обновление состояния
            obs_norm.update([next_state])
            state = next_state
            ep_reward += reward
            steps += 1
            total_samples += 1
            
            if done:
                # Добавление полного эпизода в буфер WM
                if use_wm and len(ep_states) > 1:
                    ep_states_norm = wm_obs_norm.normalize(ep_states)
                    episode_buffer.add_episode(
                        ep_states_norm,
                        ep_actions,
                        ep_rewards,
                        ep_dones
                    )
                    wm_obs_norm.update(ep_states)
                
                # Сброс эпизодных буферов
                ep_states, ep_actions, ep_rewards, ep_dones = [], [], [], []
                state, _ = env.reset(seed=seed+ep+len(real_buf['r']))

        # Обучение World Model
        if use_wm and episode_buffer and len(episode_buffer.buffer) > 1:
            # Выборка последовательностей
            s_seq, a_seq, r_seq, d_seq, ns_seq = episode_buffer.sample_sequences(
                batch_size=HP['wm_batch_size'],
                seq_length=HP['wm_horizon']
            )
            
            # Корректировка размерностей
            seq_length = HP['wm_horizon']
            s_seq = s_seq[:, :-1].flatten(0, 1)  # [batch*(seq-1), obs_dim]
            a_seq = a_seq[:, :-1].flatten(0, 1)   # [batch*(seq-1)]
            ns_seq = ns_seq[:, :seq_length-1].flatten(0, 1)  # [batch*(seq-1), obs_dim]
            
            # Перенос данных на устройство
            s_seq = s_seq.to(device)
            a_seq = a_seq.to(device)
            ns_seq = ns_seq.to(device)
            
            # Обучение World Model
            for _ in range(HP['wm_train_steps']):
                ns_pred, r_pred, d_pred = wm(s_seq, a_seq)
                
                # Расчет потерь
                state_loss = F.mse_loss(ns_pred, ns_seq)
                reward_loss = F.mse_loss(r_pred, r_seq[:, :-1].flatten(0, 1).to(device))
                done_loss = F.binary_cross_entropy(
                    d_pred, 
                    d_seq[:, :-1].flatten(0, 1).float().to(device)
                )
                
                total_loss = state_loss + reward_loss + HP['done_loss_coef']*done_loss
                
                # Оптимизация
                wm_opt.zero_grad()
                total_loss.backward()
                torch.nn.utils.clip_grad_norm_(wm.parameters(), HP['grad_clip'])
                wm_opt.step()

        # Генерация воображаемых данных
        imag_buf = {'s': [], 'a': [], 'r': [], 'd': [], 'lp': []}
        if use_wm and ep >= HP['imagination_start'] and episode_buffer:
            with torch.no_grad():
                # Выбор случайных начальных состояний из реальных данных
                init_indices = np.random.choice(
                    len(real_buf['s']), 
                    HP['wm_batch_size']//8, 
                    replace=False
                )
                init_states = [real_buf['s'][i] for i in init_indices]
                
                for s_init in init_states:
                    s = torch.FloatTensor(s_init).to(device)
                    for _ in range(HP['wm_horizon']):
                        # Выбор действия
                        logits, _ = policy(s.unsqueeze(0))
                        dist = Categorical(logits=logits)
                        action = dist.sample()
                        lp = dist.log_prob(action)
                        
                        # Предсказание WM
                        ns, r, d = wm(s.unsqueeze(0), action)
                        
                        # Сохранение перехода
                        imag_buf['s'].append(s.cpu().numpy())
                        imag_buf['a'].append(action.item())
                        imag_buf['r'].append(r.item())
                        imag_buf['d'].append(d.item() > 0.5)  # Бинаризация done
                        imag_buf['lp'].append(lp.item())
                        
                        # Обновление состояния
                        s = ns.squeeze()
                        
                        # Прерывание если done
                        if imag_buf['d'][-1]:
                            break

        # Смешивание данных
        real_ratio = HP['real_imag_ratio']
        real_samples = int(HP['batch_size'] * real_ratio)
        imag_samples = HP['batch_size'] - real_samples
        
        combined = {
            's': real_buf['s'][:real_samples] + imag_buf['s'][:imag_samples],
            'a': real_buf['a'][:real_samples] + imag_buf['a'][:imag_samples],
            'r': real_buf['r'][:real_samples] + imag_buf['r'][:imag_samples],
            'd': real_buf['d'][:real_samples] + imag_buf['d'][:imag_samples],
            'lp': real_buf['lp'][:real_samples] + imag_buf['lp'][:imag_samples]
        }
        
        # Перемешивание данных
        indices = np.random.permutation(len(combined['s']))
        for key in combined:
            combined[key] = [combined[key][i] for i in indices]
        
        # Подготовка данных для обучения
        states = torch.FloatTensor(np.array(combined['s'])).to(device)
        actions = torch.LongTensor(combined['a']).to(device)
        old_log_probs = torch.FloatTensor(combined['lp']).to(device)
        rewards_t = torch.FloatTensor(combined['r']).to(device)
        dones = torch.FloatTensor(combined['d']).to(device)
        
        # Обновление политики
        with torch.no_grad():
            _, values = policy(states)
        
        advantages, returns = compute_gae(
            values.cpu().numpy(),
            rewards_t.cpu().numpy(),
            dones.cpu().numpy()
        )
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        
        ppo_update(
            policy,
            optimizer,
            states,
            actions,
            old_log_probs,
            advantages.to(device),
            returns.to(device)
        )

        # Логирование и сохранение результатов
        rewards.append(ep_reward)
        if ep % HP['plot_interval'] == 0:
            avg = np.mean(rewards[-HP['plot_interval']:])
            print(f"Ep {ep}| Reward {ep_reward:.1f}| Avg {avg:.1f}| Steps {steps}")

    # Сохранение GIF
    if save_gif:
        frames = []
        state, _ = env.reset()
        for _ in range(HP['render_frames']):
            img = env.render()
            frames.append(img)
            s_n = obs_norm.normalize(state)
            with torch.no_grad():
                logits, _ = policy(torch.FloatTensor(s_n).to(device))
                action = logits.argmax().item()
            state, _, done, _, _ = env.step(action)
            if done:
                break
        imageio.mimsave(f'{env_name}_{"wm" if use_wm else "ppo"}_seed{seed}.gif', 
                       frames, fps=HP['gif_fps'])
    
    env.close()
    return rewards, policy, obs_norm

## Запуск экспериментов

In [ ]:
policies = []
results = {'ppo':[], 'wm':[]}

for seed in HP['seeds']:
    # Обучаем PPO
    print(f'Training PPO seed {seed}')
    r_ppo, p_ppo, obs_norm_ppo = train('LunarLander-v3', seed, use_wm=False)
    results['ppo'].append(r_ppo)
    
    # Обучаем PPO+WM
    print(f'Training PPO+WM seed {seed}')
    r_wm, p_wm, obs_norm_wm = train('LunarLander-v3', seed, use_wm=True)
    results['wm'].append(r_wm)
    
    policies.append( (p_ppo, p_wm, obs_norm_ppo, obs_norm_wm) )


Training PPO seed 42
Ep 0| Reward -994.5| Avg -994.5| Steps 512
Ep 50| Reward -1380.3| Avg -1055.2| Steps 26112
Ep 100| Reward -1109.1| Avg -1204.6| Steps 51712
Ep 150| Reward -1520.0| Avg -1302.8| Steps 77312
Ep 200| Reward -1549.2| Avg -1505.1| Steps 102912
Ep 250| Reward -1149.5| Avg -1301.6| Steps 128512
Ep 300| Reward -865.7| Avg -1406.1| Steps 154112
Ep 350| Reward -1968.3| Avg -1462.4| Steps 179712
Ep 400| Reward -1936.8| Avg -1880.1| Steps 205312
Ep 450| Reward -1491.3| Avg -1943.5| Steps 230912
Ep 500| Reward -649.0| Avg -1137.3| Steps 256512
Ep 550| Reward -1771.7| Avg -1406.8| Steps 282112
Ep 600| Reward -898.6| Avg -1573.1| Steps 307712
Ep 650| Reward -1530.3| Avg -1441.3| Steps 333312
Ep 700| Reward -2472.0| Avg -1937.5| Steps 358912
Ep 750| Reward -976.1| Avg -1535.5| Steps 384512
Ep 800| Reward -939.1| Avg -1141.8| Steps 410112
Ep 850| Reward -677.9| Avg -1202.7| Steps 435712
Ep 900| Reward -2157.1| Avg -1238.1| Steps 461312
Ep 950| Reward -1149.3| Avg -1413.1| Steps 486

## Визуализация

In [ ]:
def plot_results(r_ppo, r_wm, seeds):
    plt.figure(figsize=(12, 6))
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
    
    smooth = lambda x: np.convolve(np.pad(x, (HP['smooth_window']//2, HP['smooth_window']//2), mode='edge'), 
                                 np.ones(HP['smooth_window'])/HP['smooth_window'], mode='valid')
    
    for i, seed in enumerate(seeds):
        plt.plot(smooth(r_ppo[i]), color=colors[i], linestyle='-', alpha=0.8, label=f'PPO seed {seed}')
        plt.plot(smooth(r_wm[i]), color=colors[i], linestyle='--', linewidth=2, label=f'PPO+WM seed {seed}')
    
    plt.title('Сравнение обучения на LunarLander-v3')
    plt.xlabel('Эпизод')
    plt.ylabel('Сглаженная награда')
    plt.legend()
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig('lunar_comparison.png', dpi=120)
    plt.show()

print('Генерация графиков...')
plot_results(results['ppo'], results['wm'], HP['seeds'])

## Генератор GIF для всех политик

In [ ]:
from IPython.display import Image as IPyImage
from IPython.display import display, Image



def render_policy(env_name, policy, obs_norm, seed, filename):
    env = gym.make(env_name, render_mode='rgb_array')
    state, _ = env.reset(seed=seed)
    frames = []
    
    for _ in range(HP['render_frames']):
        img = env.render()
        frames.append(img)
        s_n = obs_norm.normalize(state)
        st = torch.FloatTensor(s_n).to(device).unsqueeze(0)
        with torch.no_grad():
            logits, _ = policy(st)
            action = logits.argmax().item()
        
        state, _, done, _, _ = env.step(action)
        if done: 
            break
    
    imageio.mimsave(filename, frames, fps=HP['gif_fps'])
    env.close()

def generate_gifs(policies, seeds):
    for (ppo_pol, wm_pol, obs_norm_ppo, obs_norm_wm), seed in zip(policies, seeds):
        # Рендерим PPO
        render_policy(
            'LunarLander-v3',
            ppo_pol,
            obs_norm_ppo,
            seed,
            f'lander_ppo_seed{seed}.gif'
        )
        print(f'lander_ppo_seed{seed}.gif')
        display(Image(filename=f'lander_ppo_seed{seed}.gif', format='gif'))
        
        # Рендерим PPO+WM
        if wm_pol is not None:
            render_policy(
                'LunarLander-v3',
                wm_pol,
                obs_norm_wm,
                seed,
                f'lander_wm_seed{seed}.gif'
            )
            print(f'lander_wm_seed{seed}.gif')
            display(Image(filename=f'lander_wm_seed{seed}.gif', format='gif'))

print('Генерация GIF...')
generate_gifs(policies, HP['seeds'])

## Выводы

### World Model не улучшает PPO в текущей реализации из-за:

- Недостаточного обучения модели (wm_train_steps=3).

- Фиксированного соотношения реальных/синтетических данных.

### Ключевые гиперпараметры:

- lr_policy=1e-4 и lr_wm=5e-5 — оптимальны.

- gamma=0.999 помогает учитывать долгосрочные награды.

- clip_eps=0.25 стабилизирует обучение.